# 🧪 Gerando Testes Unitários com LangChain e Azure ChatGPT

Notebook oficial do projeto — pensado para rodar **100% no Google Colab**,
sem pesar o disco local.

Este notebook segue as mesmas 8 etapas descritas no README do repositório:

1. Escopo e requisitos
2. Configuração do ambiente
3. Prompts e chains
4. Agentes e ferramentas
5. Estratégias de geração de testes
6. Automação e execução (pytest)
7. Ciclo TDD (Red-Green-Refactor)
8. Preparação para produção (CI/CD)


## 1️⃣ Escopo e Requisitos

Objetivo: dado um arquivo Python em `examples/`, o agente gera automaticamente `tests/test_<nome>.py`, executa com `pytest` e corrige os testes até passarem (ou até o limite de iterações).

## 2️⃣ Configuração do Ambiente

Clonamos o repositório e instalamos as dependências. Troque a URL abaixo pela do **seu** repositório depois do primeiro push.

In [ ]:
# Clone o seu repositório (troque pela URL real depois de subir para o GitHub)
!git clone https://github.com/SEU_USUARIO/langchain-test-generator.git
%cd langchain-test-generator

!pip install -q -r requirements.txt

### Configurando as credenciais com segurança (sem deixar chave hardcoded)

In [ ]:
import os
from getpass import getpass

# Provedor preferencial: tentamos Azure primeiro, com fallback automático para Gemini
os.environ["LLM_PROVIDER"] = "azure"

# --- Azure OpenAI (créditos gratuitos de trial) ---
os.environ["AZURE_OPENAI_API_KEY"] = getpass("AZURE_OPENAI_API_KEY (Enter para pular): ")
os.environ["AZURE_OPENAI_ENDPOINT"] = input("AZURE_OPENAI_ENDPOINT (Enter para pular): ")
os.environ["AZURE_OPENAI_API_VERSION"] = "2024-08-01-preview"
os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"] = input("AZURE_OPENAI_DEPLOYMENT_NAME (Enter para pular): ")

# --- Google Gemini (fallback gratuito, gere em https://aistudio.google.com/apikey) ---
os.environ["GOOGLE_API_KEY"] = getpass("GOOGLE_API_KEY (Enter para pular): ")
os.environ["GEMINI_MODEL"] = "gemini-1.5-flash"


## 3️⃣ Prompts e Chains (LCEL)

Os templates ficam em `src/prompts.py` e as chains (`Prompt | LLM | Parser`) em `src/chains.py`. Vamos instanciar o LLM (com fallback automático) e montar a chain de geração.

In [ ]:
from src.llm_provider import get_llm
from src.chains import build_test_generation_chain

llm = get_llm()
generation_chain = build_test_generation_chain(llm)
print("LLM pronto:", type(llm).__name__)

## 4️⃣ Agentes e Ferramentas do LangChain

`src/tools.py` expõe as ações do agente (ler código, escrever teste, rodar pytest). `src/agent.py` orquestra essas ferramentas em um ciclo TDD completo.

In [ ]:
from src.agent import TestGeneratorAgent

agent = TestGeneratorAgent(llm=llm)
print("Agente pronto. Máximo de iterações do ciclo TDD:", agent.max_iterations)

## 5️⃣ Estratégias de Geração de Testes

Rodando o agente para os dois exemplos do desafio: uma função simples (`calculadora.py`) e uma mais complexa (`validador_cpf.py`).

In [ ]:
resultado_calc = agent.run("calculadora")
print("Sucesso:", resultado_calc.success, "| Iterações:", resultado_calc.iterations)
print(open(resultado_calc.test_file).read())

In [ ]:
resultado_cpf = agent.run("validador_cpf")
print("Sucesso:", resultado_cpf.success, "| Iterações:", resultado_cpf.iterations)
print(open(resultado_cpf.test_file).read())

## 6️⃣ Automação e Execução dos Testes

Rodando a suíte completa com `pytest`, exatamente como o avaliador do curso fará.

In [ ]:
!pytest -v

## 7️⃣ Validação e Refatoração Contínua (Ciclo TDD)

O ciclo Red → Green → Refactor já acontece dentro de `agent.run()`: se o pytest falhar, o erro real é enviado de volta ao LLM (`build_refactor_chain`), que corrige o teste — até `max_iterations` vezes. Veja o histórico:

In [ ]:
for linha in resultado_cpf.history:
    print(linha)

## 8️⃣ Preparação para Produção

- **CI**: `.github/workflows/ci.yml` roda `pytest` + cobertura em cada push/PR.
- **CD/Deploy**: `.github/workflows/release.yml` empacota o projeto e cria uma Release do GitHub a cada tag `vX.Y.Z`.
- **Monitoramento**: `src/logger_setup.py` grava logs em `logs/run.log` e um relatório Markdown por execução.

In [ ]:
from src.logger_setup import setup_logging, write_run_report

setup_logging()
caminho_relatorio = write_run_report(resultado_cpf)
print("Relatório salvo em:", caminho_relatorio)
print(open(caminho_relatorio).read())

## 📤 Subindo para o GitHub

Após validar tudo no Colab, salve este notebook em `notebooks/` no repositório e suba as mudanças. Veja os comandos bash completos no `README.md` (seção **Como subir para o GitHub**).